# CivicLens AI — RDD2022 India YOLO11 training
Run this notebook in **Google Colab with a GPU runtime**. It downloads the published RDD2022 archive, prepares the labelled India images, fine-tunes YOLO11n, evaluates a held-out split, and exports ONNX. It does not claim NMC integration or Nagpur field validation.

**Before starting:** Runtime → Change runtime type → T4 GPU (or better). Training may take several hours depending on GPU and epochs.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/ayush0531/civiclens-ai.git
%cd civiclens-ai
!git checkout main
!pip install -q -r ml/requirements.txt

## Download the published RDD2022 archive
The archive contains labelled training images and unlabelled challenge test images. We use only labelled India images for the train/validation/held-out split; we do not report metrics on unlabelled challenge images. The archive is large; ensure your Colab session has enough disk space.

In [ ]:
from pathlib import Path
import urllib.request, zipfile

archive = Path('/content/RDD2022_full.zip')
url = 'https://ndownloader.figshare.com/files/38030910'
if not archive.exists():
    print('Downloading RDD2022 archive (about 12.7 GB)...')
    urllib.request.urlretrieve(url, archive)
print(f'Archive size: {archive.stat().st_size / (1024**3):.2f} GiB')
india_zip = Path('/content/India.zip')
with zipfile.ZipFile(archive) as z:
    member = next((n for n in z.namelist() if n.lower().endswith('/india.zip') or n.lower() == 'india.zip'), None)
    if member is None: raise RuntimeError('India.zip was not found inside the RDD2022 archive.')
    with z.open(member) as src, india_zip.open('wb') as dst:
        import shutil; shutil.copyfileobj(src, dst)
india_root = Path('/content/RDD2022_India')
if not india_root.exists():
    with zipfile.ZipFile(india_zip) as z: z.extractall(india_root)
xml_count = len(list(india_root.rglob('*.xml')))
image_count = sum(1 for p in india_root.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png'})
print(f'India images: {image_count}; XML annotations: {xml_count}')
if xml_count < 100: raise RuntimeError('Too few labelled India XML annotations were extracted; inspect the archive.')

In [ ]:
from pathlib import Path
root = Path('/content/RDD2022_India')
xmls = list(root.rglob('*.xml'))
print('India XML annotations:', len(xmls))
if len(xmls) < 100: raise RuntimeError('Too few India XML annotations were extracted.')
!python ml/prepare_rdd2022.py --source /content/RDD2022_India --output /content/datasets/rdd2022-india

## Train YOLO11n
Default: 100 epochs, 640px images. For a short smoke test, set `EPOCHS = 2`; do not report smoke-test metrics as final model performance.

In [ ]:
EPOCHS = 100
BATCH = 16
!python ml/train_yolo11.py --data /content/datasets/rdd2022-india/data.yaml --epochs {EPOCHS} --imgsz 640 --batch {BATCH} --device 0 --workers 8

## Review and package the verified outputs
The training script checks that best.pt exists, evaluates the held-out test split, exports ONNX, loads it with ONNX Runtime, and runs a tensor-shape smoke test. Review the confusion matrix, PR curve, per-class metrics, and example predictions before using the model in production. The held-out split is an internal RDD2022 India split, not independent Nagpur field validation.

In [ ]:
from pathlib import Path
import json
from IPython.display import display, Image
runs = Path('/content/civiclens-ai/runs')
summaries = sorted(runs.rglob('evaluation_summary.json'), key=lambda p: p.stat().st_mtime)
if not summaries:
    raise RuntimeError('No evaluation_summary.json found. Training/evaluation/export did not finish successfully.')
summary_path = summaries[-1]
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
run_dir = summary_path.parent
for name in ['confusion_matrix.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png', 'results.png']:
    p = run_dir / name
    if p.exists():
        print('Plot available:', p)
        display(Image(filename=str(p)))
print('Run directory:', run_dir)


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files
artifact_zip = Path('/content/CivicLens_YOLO11_Training_Artifacts.zip')
with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in run_dir.rglob('*'):
        if p.is_file() and p.suffix.lower() in {'.pt', '.onnx', '.json', '.csv', '.png'}:
            z.write(p, p.relative_to(run_dir.parent))
print('Packaged training checkpoints, ONNX, evaluation summary, logs and plots:', artifact_zip)
files.download(str(artifact_zip))
